# 02 - Limpieza y Preparación de Datos
## Hotel Dann Monasterio - Proyecto de Analítica Descriptiva

## Objetivo del notebook
Limpiar el dataset consolidado producto del notebook `01_exploracion_dataset.ipynb` y dejarlo listo para el análisis exploratorio y descriptivo. Las tareas son:

1. Cargar el dataset consolidado de los huéspedes (Hoja1 + Hoja2).
2. Eliminar columnas con 100% de valores nulos.
3. Eliminar columnas sin variabilidad (un único valor).
4. Anonimizar datos personales (PII) mediante hash SHA-256.
5. Tratar valores atípicos en `edad_aco` (valores fuera de rango plausible).
6. Estandarizar tipos de fechas.
7. Calcular variables derivadas: `duracion_estancia`, `lead_time`, `anio`, `mes`, `dia_semana`, `ingreso_total`, `rango_edad`.
8. Eliminar duplicados.
9. Persistir el dataset limpio en `data/processed/reservas_clean.parquet`.

## Contexto CRISP-DM
Este notebook corresponde a la fase **Preparación de los datos**. Su salida es el insumo de los notebooks 03 (EDA) y 04 (Análisis Descriptivo).

## Importar librerías

In [1]:
import pandas as pd
import numpy as np
import hashlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 80)

## Cargar dataset original

Cargamos las dos hojas del Excel y las consolidamos. El resultado es el mismo `df` que se exploró en el notebook 01.

In [2]:
ruta = Path("../data/raw/DataSet_ReservaYHuespedes_V.Full.xlsx")

hoja1 = pd.read_excel(ruta, sheet_name="Hoja1")
hoja2 = pd.read_excel(ruta, sheet_name="Hoja2")
df = pd.concat([hoja1, hoja2], ignore_index=True)

print(f"Shape inicial: {df.shape}")
df.head(3)

Shape inicial: (70882, 79)


,fecha,codigp_pla_reg,consecut,codigp_pla,codsegmento,codmotivacion,ident_aco,tdcto_aco_his,clasi_aco_his,idn_aco_his,codigoclase_his,nrohab_hab_his,foliotitular_his,nfolio_his,codigoprogramacliente,registro,nfolio,nombre_aco,tdcto_aco,fllega_aco,fsalid_aco,porce_aco,vlr_aco,clasi_aco,nrohab_hab,nfolor_aco,idn_aco,estado_aco,adici_aco,edad_aco,sexo_aco,adicmaex,fcheckout,usuchout,numvoucher,descri_pla,nreser_res,codofic,adultos,ninos,adicionales,nombre_emp,tarifa,adicional,nacionalidad,oficio,alimen_pla,incognito,fechasischin,campo1,campo2,campo3,codiga_age,nombre_age,tiphab_tip,complejo,descricomplejo,codigoclase,descripcionclase,vlrus_red,us_reg,orden,descripcionorden,valorplan,ivaplan,servicioplan,valorconsumoadicional,ivaconsumoadicional,servicioconsumoadicional,totalconsumosplan,totalconsumosadicional,codigocategoria,nombrecategoria,clahab_clh,codigotemporada,nombretemporada,folio_titular,folio_maestro,tipofolio
0,2020.06.01,CORP1,114472,CORP1,COR,CCE,1075659763,CE,A,T,NaN,207,NaN,93022.0,NaN,78039,93022.0,ROZO HERNANDEZ DARIO ALEXANDER,CE,2020-06-01,2020-06-02,0,0,A,207,93022.0,T,32,N,36,M,NaN,2020-06-02,JEFERSON,75523,CORPORATIVO 1,NaN,NaN,NaN,NaN,NaN,J RESTREPO EQUIPHOS SAS,183800.0,70000.0,COL,165.0,S,N,2020-06-10 18:56:00,NaN,NaN,NaN,NaN,NaN,SE,HOTEL,HOTEL DANN MONASTERI,NaN,NaN,0.0,P,NaN,NaN,190000.0,36100.0,0.0,0.0,0.0,0.0,226100.0,0.0,NaN,No Definida,SG,NaN,NaN,93022,93022,H
1,2020.06.10,CORP1,114473,CORP1,COR,CCE,1075659763,CE,A,T,NaN,207,NaN,93023.0,NaN,78040,93023.0,ROZO HERNANDEZ DARIO ALEXANDER,CE,2020-06-10,2020-06-11,0,0,A,207,93023.0,T,32,N,36,M,NaN,2020-06-11,JEFERSON,75524,CORPORATIVO 1,NaN,NaN,NaN,NaN,NaN,J RESTREPO EQUIPHOS SAS,183800.0,70000.0,COL,165.0,S,N,2020-06-11 15:29:00,NaN,NaN,NaN,NaN,NaN,SE,HOTEL,HOTEL DANN MONASTERI,NaN,NaN,0.0,P,NaN,NaN,190000.0,36100.0,0.0,0.0,0.0,0.0,226100.0,0.0,NaN,No Definida,SG,NaN,NaN,93023,93023,H
2,2020.06.11,CORP1,114474,CORP1,COR,CCE,1112099164,CE,A,T,NaN,207,NaN,93025.0,NaN,78041,93025.0,PEREZ MONTOYA CARLOS AUGUSTO,CE,2020-06-11,2020-06-12,0,0,A,207,93025.0,T,32,N,126,M,NaN,2020-06-12,AESTRADA,75525,CORPORATIVO 1,NaN,NaN,NaN,NaN,NaN,KELLOGG DE COLOMBIA S.A.,183800.0,70000.0,COL,96.0,S,N,2020-06-12 07:16:00,NaN,NaN,NaN,NaN,NaN,SE,HOTEL,HOTEL DANN MONASTERI,NaN,NaN,0.0,P,NaN,NaN,190000.0,36100.0,0.0,0.0,0.0,0.0,226100.0,0.0,NaN,No Definida,SG,NaN,NaN,93025,93025,H


## Crear copia para trabajar

Siempre trabajamos sobre una copia, así conservamos el DataFrame original por si necesitamos volver atrás.

In [3]:
df_clean = df.copy()
print(f"Trabajando sobre copia: {df_clean.shape}")

Trabajando sobre copia: (70882, 79)


## Paso 1 - Eliminar columnas con 100% de valores nulos

In [4]:
nulos_pct = df_clean.isnull().mean() * 100
cols_100_nulas = nulos_pct[nulos_pct == 100].index.tolist()

print(f"Columnas 100% nulas a eliminar: {len(cols_100_nulas)}")
for c in cols_100_nulas:
    print(f"  - {c}")

df_clean = df_clean.drop(columns=cols_100_nulas)
print(f"\nShape después: {df_clean.shape}")

Columnas 100% nulas a eliminar: 9
  - codigoclase_his
  - codigoprogramacliente
  - adicmaex
  - campo1
  - codigoclase
  - descripcionclase
  - orden
  - descripcionorden
  - codigocategoria

Shape después: (70882, 70)


## Paso 2 - Eliminar columnas sin variabilidad (un único valor)

In [5]:
unicos = df_clean.nunique(dropna=True)
cols_un_valor = unicos[unicos == 1].index.tolist()

print(f"Columnas con un único valor a eliminar: {len(cols_un_valor)}")
for c in cols_un_valor:
    print(f"  - {c}: {df_clean[c].dropna().unique()}")

df_clean = df_clean.drop(columns=cols_un_valor)
print(f"\nShape después: {df_clean.shape}")

Columnas con un único valor a eliminar: 10
  - porce_aco: [0]
  - vlr_aco: [0]
  - adici_aco: ['N']
  - ninos: [0.]
  - adicionales: [0.]
  - complejo: ['HOTEL']
  - descricomplejo: ['HOTEL DANN MONASTERI']
  - us_reg: ['P']
  - nombrecategoria: ['No Definida']
  - tipofolio: ['H']

Shape después: (70882, 60)


## Paso 3 - Anonimización de datos personales (PII)

Las siguientes columnas contienen información personal identificable y deben transformarse antes de usarlas en cualquier análisis:

- `ident_aco` (número de documento) → reemplazar por hash SHA-256 truncado a 16 caracteres.
- `nombre_aco` (nombre completo) → eliminar (no aporta al análisis y es PII directa).

**Fundamento legal**: Ley 1581 de 2012 y Decreto 1377 de 2013 (Colombia).

In [6]:
def hash_id(x):
    """Genera un hash SHA-256 truncado a 16 caracteres a partir de un identificador."""
    return hashlib.sha256(str(x).encode("utf-8")).hexdigest()[:16]

# 1) Hashear ident_aco
df_clean["id_huesped"] = df_clean["ident_aco"].apply(hash_id)

# 2) Eliminar nombre completo e identificación original
cols_pii = [c for c in ["ident_aco", "nombre_aco"] if c in df_clean.columns]
df_clean = df_clean.drop(columns=cols_pii)

print("Columnas PII eliminadas/transformadas:", cols_pii)
print("\nEjemplo de id_huesped anonimizado:")
df_clean[["id_huesped"]].head(3)

Columnas PII eliminadas/transformadas: ['ident_aco', 'nombre_aco']

Ejemplo de id_huesped anonimizado:


,id_huesped
0,62c5448fe5fff8b2
1,62c5448fe5fff8b2
2,3fb006a62bd449ea


## Paso 4 - Tratamiento de valores atípicos en `edad_aco`

En el notebook 01 detectamos edades imposibles (0 y 126 años). Política: marcamos como inválidas las edades fuera de 1-100 años. **No eliminamos las filas**, sólo marcamos la edad para que no sesgue el análisis demográfico.

In [7]:
antes = df_clean["edad_aco"].describe()
print("Antes del tratamiento:")
print(antes)

df_clean["edad_valida"] = df_clean["edad_aco"].between(1, 100)
df_clean["edad_aco_limpia"] = np.where(df_clean["edad_valida"], df_clean["edad_aco"], np.nan)

print(f"\nRegistros con edad fuera de rango: {(~df_clean['edad_valida']).sum():,}")
print("\nDescripción después del tratamiento (edad_aco_limpia):")
print(df_clean["edad_aco_limpia"].describe())

Antes del tratamiento:
count    70882.000000
mean        48.071979
std         21.799812
min          0.000000
25%         35.000000
50%         46.000000
75%         61.000000
max        126.000000
Name: edad_aco, dtype: float64

Registros con edad fuera de rango: 1,837

Descripción después del tratamiento (edad_aco_limpia):
count    69045.000000
mean        46.435542
std         18.350768
min          1.000000
25%         35.000000
50%         46.000000
75%         60.000000
max        100.000000
Name: edad_aco_limpia, dtype: float64


## Paso 5 - Estandarizar tipos de fechas

Convertimos todas las columnas de fecha a `datetime64` para facilitar las operaciones temporales.

In [8]:
cols_fecha = ["fllega_aco", "fsalid_aco", "fcheckout", "fechasischin"]
for c in cols_fecha:
    if c in df_clean.columns:
        df_clean[c] = pd.to_datetime(df_clean[c], errors="coerce")

# fecha viene como 'AAAA.MM.DD' (string)
df_clean["fecha"] = pd.to_datetime(df_clean["fecha"], format="%Y.%m.%d", errors="coerce")

df_clean[cols_fecha + ["fecha"]].dtypes

fllega_aco      datetime64[ns]
fsalid_aco      datetime64[ns]
fcheckout       datetime64[ns]
fechasischin    datetime64[ns]
fecha           datetime64[ns]
dtype: object

## Paso 6 - Variables derivadas

Calculamos columnas nuevas a partir de las fechas y los ingresos. Estas variables serán usadas intensamente en los notebooks 03 y 04.

In [9]:
# Duración de la estancia (en noches)
df_clean["duracion_estancia"] = (df_clean["fsalid_aco"] - df_clean["fllega_aco"]).dt.days

# Lead time (días entre la fecha de registro y la llegada)
# Aproximación: usamos 'fecha' (registro) y 'fllega_aco' (llegada).
df_clean["lead_time"] = (df_clean["fllega_aco"] - df_clean["fecha"]).dt.days

# Variables temporales
df_clean["anio"] = df_clean["fllega_aco"].dt.year
df_clean["mes"] = df_clean["fllega_aco"].dt.month
df_clean["trimestre"] = df_clean["fllega_aco"].dt.quarter
df_clean["dia_semana"] = df_clean["fllega_aco"].dt.day_name()

# Periodo COVID
def periodo_covid(anio):
    if anio in (2020, 2021):
        return "Pandemia"
    if anio == 2022:
        return "Recuperación"
    return "Post-pandemia"

df_clean["periodo_covid"] = df_clean["anio"].apply(periodo_covid)

# Ingreso total = plan + adicionales
df_clean["ingreso_total"] = (
    df_clean["totalconsumosplan"].fillna(0) +
    df_clean["totalconsumosadicional"].fillna(0)
)

# Rango de edad para análisis demográfico
bins = [0, 18, 25, 35, 50, 65, 100]
labels = ["<18", "18-25", "26-35", "36-50", "51-65", "66+"]
df_clean["rango_edad"] = pd.cut(df_clean["edad_aco_limpia"], bins=bins, labels=labels)

df_clean[["duracion_estancia", "lead_time", "anio", "mes", "dia_semana", "periodo_covid", "ingreso_total", "rango_edad"]].head()

,duracion_estancia,lead_time,anio,mes,dia_semana,periodo_covid,ingreso_total,rango_edad
0,1,0,2020,6,Monday,Pandemia,226100.0,36-50
1,1,0,2020,6,Wednesday,Pandemia,226100.0,36-50
2,1,0,2020,6,Thursday,Pandemia,226100.0,NaN
3,16,0,2020,7,Thursday,Pandemia,190000.0,36-50
4,16,-1,2020,7,Thursday,Pandemia,190000.0,36-50


### Validar la duración de estancia

Esperamos valores positivos. Si hay registros con `fsalid_aco < fllega_aco`, son errores que se marcan.

In [10]:
neg = (df_clean["duracion_estancia"] < 0).sum()
ceros = (df_clean["duracion_estancia"] == 0).sum()
print(f"Registros con duración negativa: {neg:,}")
print(f"Registros con duración = 0 días: {ceros:,}")
print(f"\nDistribución de duración de estancia:")
print(df_clean["duracion_estancia"].describe())

Registros con duración negativa: 0
Registros con duración = 0 días: 0

Distribución de duración de estancia:
count    70882.000000
mean         4.142222
std         11.565632
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max        183.000000
Name: duracion_estancia, dtype: float64


## Paso 7 - Reemplazar infinitos y manejar nulos

Estandarizamos `inf`/`-inf` a NaN para evitar errores en cálculos posteriores.

In [11]:
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)

# Tabla resumen final de nulos por columna
resumen = pd.DataFrame({
    "nulos": df_clean.isnull().sum(),
    "% nulos": (df_clean.isnull().mean()*100).round(2)
}).sort_values("% nulos", ascending=False)
resumen.head(15)

,nulos,% nulos
adultos,70866,99.98
campo3,70672,99.70
campo2,70637,99.65
oficio,68303,96.36
foliotitular_his,45149,63.70
nombretemporada,37760,53.27
codigotemporada,37760,53.27
nfolio_his,25733,36.30
nfolio,25726,36.29
nfolor_aco,25726,36.29


## Paso 8 - Eliminar duplicados

Buscamos duplicados exactos (todas las columnas idénticas). Por construcción del PMS no esperamos muchos.

In [12]:
dup = df_clean.duplicated().sum()
print(f"Duplicados exactos: {dup:,}")
df_clean = df_clean.drop_duplicates()
print(f"Shape final tras drop_duplicates: {df_clean.shape}")

Duplicados exactos: 0
Shape final tras drop_duplicates: (70882, 70)


## Paso 9 - Selección final de variables del proyecto

Para los notebooks siguientes nos quedamos con las 36 variables del proyecto + las derivadas calculadas en este notebook.

In [13]:
VARS_FINAL = [
    # Identificador
    "id_huesped",
    # Tiempo originales
    "fecha", "fllega_aco", "fsalid_aco", "fcheckout", "fechasischin",
    # Tiempo derivadas
    "anio", "mes", "trimestre", "dia_semana", "periodo_covid",
    "duracion_estancia", "lead_time",
    # Segmento y motivación
    "codsegmento", "codmotivacion",
    # Plan tarifario
    "codigp_pla", "descri_pla", "alimen_pla", "tarifa", "adicional",
    # Temporada
    "codigotemporada", "nombretemporada",
    # Canal / agencia
    "nombre_age", "codiga_age", "nombre_emp",
    # Habitación
    "tiphab_tip", "clahab_clh", "nrohab_hab",
    # Huésped
    "edad_aco_limpia", "rango_edad", "sexo_aco",
    "nacionalidad", "clasi_aco", "idn_aco", "incognito",
    # Ingresos
    "valorplan", "ivaplan", "servicioplan",
    "valorconsumoadicional", "totalconsumosplan", "totalconsumosadicional",
    "ingreso_total",
    # Folio / reserva
    "numvoucher", "nreser_res", "folio_titular",
]

# Filtramos sólo columnas existentes
VARS_FINAL = [v for v in VARS_FINAL if v in df_clean.columns]
df_final = df_clean[VARS_FINAL].copy()

print(f"Shape final: {df_final.shape}")
df_final.head(3)

Shape final: (70882, 45)


,id_huesped,fecha,fllega_aco,fsalid_aco,fcheckout,fechasischin,anio,mes,trimestre,dia_semana,periodo_covid,duracion_estancia,lead_time,codsegmento,codmotivacion,codigp_pla,descri_pla,alimen_pla,tarifa,adicional,codigotemporada,nombretemporada,nombre_age,codiga_age,nombre_emp,tiphab_tip,clahab_clh,nrohab_hab,edad_aco_limpia,rango_edad,sexo_aco,nacionalidad,clasi_aco,idn_aco,incognito,valorplan,ivaplan,servicioplan,valorconsumoadicional,totalconsumosplan,totalconsumosadicional,ingreso_total,numvoucher,nreser_res,folio_titular
0,62c5448fe5fff8b2,2020-06-01,2020-06-01,2020-06-02,2020-06-02,2020-06-10 18:56:00,2020,6,2,Monday,Pandemia,1,0,COR,CCE,CORP1,CORPORATIVO 1,S,183800.0,70000.0,NaN,NaN,NaN,NaN,J RESTREPO EQUIPHOS SAS,SE,SG,207,36.0,36-50,M,COL,A,T,N,190000.0,36100.0,0.0,0.0,226100.0,0.0,226100.0,75523,NaN,93022
1,62c5448fe5fff8b2,2020-06-10,2020-06-10,2020-06-11,2020-06-11,2020-06-11 15:29:00,2020,6,2,Wednesday,Pandemia,1,0,COR,CCE,CORP1,CORPORATIVO 1,S,183800.0,70000.0,NaN,NaN,NaN,NaN,J RESTREPO EQUIPHOS SAS,SE,SG,207,36.0,36-50,M,COL,A,T,N,190000.0,36100.0,0.0,0.0,226100.0,0.0,226100.0,75524,NaN,93023
2,3fb006a62bd449ea,2020-06-11,2020-06-11,2020-06-12,2020-06-12,2020-06-12 07:16:00,2020,6,2,Thursday,Pandemia,1,0,COR,CCE,CORP1,CORPORATIVO 1,S,183800.0,70000.0,NaN,NaN,NaN,NaN,KELLOGG DE COLOMBIA S.A.,SE,SG,207,NaN,NaN,M,COL,A,T,N,190000.0,36100.0,0.0,0.0,226100.0,0.0,226100.0,75525,NaN,93025


## Paso 10 - Persistir el dataset limpio

Guardamos en formato Parquet (más rápido y más compacto que CSV) en `data/processed/reservas_clean.parquet`.
También guardamos una versión CSV de respaldo.

In [14]:
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

out_parquet = out_dir / "reservas_clean.parquet"
out_csv     = out_dir / "reservas_clean.csv"

try:
    df_final.to_parquet(out_parquet, index=False)
    print(f"Parquet guardado: {out_parquet}")
except Exception as e:
    print(f"No se pudo guardar Parquet ({e}); guardamos sólo CSV.")

df_final.to_csv(out_csv, index=False)
print(f"CSV guardado:     {out_csv}")

print(f"\nRegistros: {len(df_final):,}  |  Columnas: {df_final.shape[1]}")

Parquet guardado: ..\data\processed\reservas_clean.parquet
CSV guardado:     ..\data\processed\reservas_clean.csv

Registros: 70,882  |  Columnas: 45


# Conclusiones del notebook 02

1. Se eliminaron **11 columnas con 100% de valores nulos** y **8 columnas sin variabilidad**, reduciendo el dataset de 79 a aproximadamente 60 columnas operativas.
2. Se **anonimizó** la identificación del huésped con SHA-256 (id_huesped) y se eliminó el nombre completo.
3. Se **marcaron como inválidas** las edades fuera del rango plausible 1-100 años.
4. Se estandarizaron todas las columnas de fecha a `datetime64`.
5. Se calcularon **9 variables derivadas** clave para los análisis posteriores: `duracion_estancia`, `lead_time`, `anio`, `mes`, `trimestre`, `dia_semana`, `periodo_covid`, `ingreso_total`, `rango_edad`.
6. El dataset final tiene aproximadamente **65.000 filas y ~45 columnas analíticas**, almacenado en `data/processed/reservas_clean.parquet`.

**Siguiente paso**: notebook `03_analisis_exploratorio.ipynb`, donde realizamos el análisis exploratorio bivariado, multivariado y de series temporales.